## Setup & Imports

In [4]:
%pip install pandas numpy torch scikit-learn scipy
import os
import sys
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, pairwise_distances
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM, SVC
from sklearn.neighbors import LocalOutlierFactor, KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings("ignore")

BASE_DIR = os.path.abspath(".")
DATASET_DIR = os.path.join(BASE_DIR, "Gearbox Dataset")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

WINDOWS = [300, 400, 500, 600, 700, 800, 900, 1000]
SEED = 42

sys.path.insert(0, os.path.join(BASE_DIR, "efficient_kan"))
from efficient_kan import KAN


  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached torch-2.11.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_bindings-13.2.0-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached nvidia_cudnn_cu13-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusp

## 1) Feature Extraction (44 Features)

In [5]:
# 1) Import all the data from the 4 sensors and calculate the 11 statistical features for all of them.
import glob
from scipy.stats import skew, kurtosis

FEATURE_NAMES = ["mean", "rms", "std", "var", "skew", "kurtosis", "p2p", "crest", "shape", "margin", "impulse"]
CHANNELS = ["S1", "S2", "S3", "S4"]

def compute_11_features(signal):
    rms_val = np.sqrt(np.mean(signal**2))
    margin_denom = np.mean(np.sqrt(np.abs(signal)))**2
    return [
        np.mean(signal), rms_val, np.std(signal), np.var(signal), skew(signal), kurtosis(signal),
        np.ptp(signal), np.max(np.abs(signal)) / rms_val if rms_val > 0 else 0,
        rms_val / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) > 0 else 0,
        np.max(np.abs(signal)) / margin_denom if margin_denom > 0 else 0,
        np.max(np.abs(signal)) / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) > 0 else 0
    ]

def extract_features(W):
    all_data = []
    filepaths = glob.glob(os.path.join(DATASET_DIR, "**", "*.txt"), recursive=True)
    for filepath in filepaths:
        fname = os.path.basename(filepath).lower()
        if "healthy" in fname: label = 0
        elif "broken" in fname: label = 1
        else: continue
            
        load_val = 0
        for l in [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]:
            if f"{l}hz" in fname or f"{l}load" in fname: load_val = l
                
        try:
            df = pd.read_csv(filepath, sep='\t', header=None)
            if df.shape[1] < 4: df = pd.read_csv(filepath, sep=',', header=None)
            series = df.iloc[:, :4].values
        except: continue
            
        for start in range(0, series.shape[0] - W + 1, W):
            row = []
            for c in range(4): row.extend(compute_11_features(series[start:start+W, c]))
            row.extend([load_val, label])
            all_data.append(row)
            
    cols = [f"{c}_{f}" for c in CHANNELS for f in FEATURE_NAMES] + ["load", "label"]
    pd.DataFrame(all_data, columns=cols).to_csv(os.path.join(PROCESSED_DIR, f"features_W{W}.csv"), index=False)

for W in WINDOWS:
    if not os.path.exists(os.path.join(PROCESSED_DIR, f"features_W{W}.csv")):
        extract_features(W)


## 2) Supervised Models Sweep (W=300 to 1000) on 44 Features

In [6]:
# 2) Sweep supervised models (including KAN) from W = 300 to 1000 using the 44 feature vectors
from IPython.display import display

def get_supervised_models(): return {"DT": DecisionTreeClassifier(random_state=SEED), "RF": RandomForestClassifier(n_estimators=100, random_state=SEED), "SVM": SVC(kernel="rbf", random_state=SEED), "NB": GaussianNB(), "KNN": KNeighborsClassifier(n_neighbors=5), "GBC": GradientBoostingClassifier(n_estimators=100, random_state=SEED), "LR": LogisticRegression(max_iter=1000, random_state=SEED)}

def train_kan_supervised(X_tr, y_tr, X_te, y_te):
    model = KAN(layers_hidden=[X_tr.shape[1], max(4, X_tr.shape[1]//2), 2], grid_size=5, spline_order=3)
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long)), batch_size=512, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for _ in range(30):
        for bx, by in loader:
            opt.zero_grad()
            crit(model(bx), by).backward()
            opt.step()
    model.eval()
    with torch.no_grad(): return accuracy_score(y_te, torch.argmax(model(torch.tensor(X_te, dtype=torch.float32)), dim=1).numpy()) * 100

results_44F = pd.DataFrame(index=list(get_supervised_models().keys()) + ["KAN"], columns=WINDOWS, dtype=float)

for W in WINDOWS:
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f"features_W{W}.csv"))
    X, y = df.drop(columns=["label", "load"]).values, df["label"].values
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    fold_accs = {m: [] for m in results_44F.index}
    
    for tr, te in skf.split(X, y):
        sc = MinMaxScaler(); X_tr, X_te = sc.fit_transform(X[tr]), sc.transform(X[te])
        models = get_supervised_models()
        for mname, clf in models.items():
            clf.fit(X_tr, y[tr])
            fold_accs[mname].append(accuracy_score(y[te], clf.predict(X_te)) * 100)
        fold_accs["KAN"].append(train_kan_supervised(X_tr, y[tr], X_te, y[te]))
        
    for m in results_44F.index: results_44F.loc[m, W] = np.mean(fold_accs[m])

print("--- Supervised Models Accuracy (%) [44 Features] ---")
display(results_44F.round(2))


--- Supervised Models Accuracy (%) [44 Features] ---


,300,400,500,600,700,800,900,1000
DT,96.34,96.93,97.32,97.38,97.81,98.37,98.52,98.86
RF,98.65,99.01,99.31,99.52,99.65,99.64,99.78,99.85
SVM,99.15,99.76,99.88,99.97,100.00,100.00,100.00,100.00
NB,86.90,89.71,92.13,94.76,95.79,97.50,98.84,99.20
KNN,97.89,98.91,99.26,99.29,99.37,99.84,99.96,99.90
GBC,98.93,99.25,99.33,99.49,99.62,99.36,99.60,99.55
LR,98.74,99.44,99.70,99.85,99.97,100.00,100.00,100.00
KAN,98.71,98.99,98.86,98.72,97.46,98.17,96.29,97.66


## 3) Two-NN Intrinsic Dimension Calculation at W=1000

In [11]:
# 3) Calculate 2-NN at W = 300 and W = 1000
for W in [300, 1000]:
    # Notice the f"features_W{W}.csv" dynamically loads either 300 or 1000 based on the loop
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f"features_W{W}.csv"))
    
    # Isolate healthy data and drop meta columns
    X_h = StandardScaler().fit_transform(df[df['label'] == 0].drop(columns=['load', 'label']).values)
    nn_twonn = NearestNeighbors(n_neighbors=3, metric='euclidean', n_jobs=-1).fit(X_h)
    dists, _ = nn_twonn.kneighbors(X_h)
    r1, r2 = dists[:, 1], dists[:, 2]
    valid = r1 > 1e-10
    
    mu = r2[valid] / r1[valid]
    id_est = len(mu) / np.sum(np.log(mu))
    
    print(f"TWO-NN Intrinsic Dimension for W={W}: {id_est:.2f}")

TWO-NN Intrinsic Dimension for W=300: 13.01
TWO-NN Intrinsic Dimension for W=1000: 11.74


## 4) 100-Epoch KAN for Feature Importance (Top 13 Vectors)

In [8]:
# 4) Run 100-epoch KAN on W = 1000 to find true feature importance
df = pd.read_csv(os.path.join(PROCESSED_DIR, f"features_W1000.csv"))
feat_cols = list(df.drop(columns=["label", "load"]).columns)
X_s = MinMaxScaler().fit_transform(df[feat_cols].values)
y = df["label"].values

n_f = X_s.shape[1]
model = KAN(layers_hidden=[n_f, n_f//2, 2], grid_size=5, spline_order=3)
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.tensor(X_s, dtype=torch.float32), torch.tensor(y, dtype=torch.long)), batch_size=512, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

model.train()
for _ in range(100):
    for bx, by in loader:
        opt.zero_grad()
        crit(model(bx), by).backward()
        opt.step()

score = model.layers[0].spline_weight.detach().cpu().abs().mean(dim=(0, 2)).numpy()
imp_df = pd.DataFrame({"Feature": feat_cols, "Importance": score}).sort_values("Importance", ascending=False).reset_index(drop=True)

TOP_13_VECTORS = imp_df.head(13)["Feature"].tolist()
print("--- Top 13 Most Important Feature Vectors ---")
print(TOP_13_VECTORS)


--- Top 13 Most Important Feature Vectors ---
['S1_std', 'S1_rms', 'S1_var', 'S4_mean', 'S2_skew', 'S2_shape', 'S1_shape', 'S1_p2p', 'S4_shape', 'S3_mean', 'S2_std', 'S3_shape', 'S2_rms']


## 5) Unsupervised Anomaly Detection Sweep (44 vs 13 Features)

In [9]:
# 5) Run The 9 unsupervised anomaly detection algorithms on 44 vectors vs 13 vectors across W=300-1000
class AE(nn.Module):
    def __init__(self, d): super().__init__(); self.e = nn.Sequential(nn.Linear(d,32), nn.ReLU(), nn.Linear(32,16), nn.ReLU(), nn.Linear(16,8), nn.ReLU()); self.d = nn.Sequential(nn.Linear(8,16), nn.ReLU(), nn.Linear(16,32), nn.ReLU(), nn.Linear(32,d))
    def forward(self, x): return self.d(self.e(x))
class VAE(nn.Module):
    def __init__(self, d): super().__init__(); self.e = nn.Sequential(nn.Linear(d,32), nn.ReLU(), nn.Linear(32,16), nn.ReLU()); self.mu = nn.Linear(16,8); self.var = nn.Linear(16,8); self.d = nn.Sequential(nn.Linear(8,16), nn.ReLU(), nn.Linear(16,32), nn.ReLU(), nn.Linear(32,d))
    def forward(self, x): h=self.e(x); m, lv=self.mu(h), self.var(h); return self.d(m + torch.randn_like(torch.exp(0.5*lv))*torch.exp(0.5*lv)), m, lv
class DeepSVDD(nn.Module):
    def __init__(self, d): super().__init__(); self.n = nn.Sequential(nn.Linear(d,32), nn.ReLU(), nn.Linear(32,16), nn.ReLU(), nn.Linear(16,16))
    def forward(self, x): return self.n(x)
class TSNetwork(nn.Module):
    def __init__(self, d): super().__init__(); self.n = nn.Sequential(nn.Linear(d,32), nn.ReLU(), nn.Linear(32,16))
    def forward(self, x): return self.n(x)
class DAE(nn.Module):
    def __init__(self, d): super().__init__(); self.e=nn.Sequential(nn.Linear(d,32), nn.ReLU(), nn.Linear(32,8)); self.d=nn.Sequential(nn.Linear(8,32), nn.ReLU(), nn.Linear(32,d))
    def forward(self, x): return self.d(self.e(x))

def k_center(f, frac=0.1):
    c = [np.random.randint(0, f.shape[0])]; dists = pairwise_distances(f, f[c], metric='euclidean').flatten()
    for _ in range(max(1, int(f.shape[0]*frac)) - 1): idx=np.argmax(dists); c.append(idx); dists=np.minimum(dists, pairwise_distances(f, f[[idx]]).flatten())
    return f[c]

def run_unsup(fts):
    res = pd.DataFrame(index=["IsolationForest", "OC-SVM", "LOF", "PatchCore", "Autoencoder", "VAE", "DeepSVDD", "TeacherStudent", "SSL_DAE"], columns=WINDOWS, dtype=float)
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    for W in WINDOWS:
        df = pd.read_csv(os.path.join(PROCESSED_DIR, f"features_W{W}.csv"))
        tr, te = train_test_split(df[df["label"]==0], test_size=0.3, random_state=SEED)
        te = pd.concat([te, df[df["label"]==1]])
        X_tr, X_te = StandardScaler().fit(tr[fts].values).transform(tr[fts].values), StandardScaler().fit(tr[fts].values).transform(te[fts].values)
        y_te = te["label"].values
        
        res.loc["IsolationForest", W] = roc_auc_score(y_te, -IsolationForest(n_estimators=100, random_state=SEED).fit(X_tr).decision_function(X_te))
        res.loc["OC-SVM", W] = roc_auc_score(y_te, -OneClassSVM(kernel="rbf", gamma="scale", nu=0.05).fit(X_tr).decision_function(X_te))
        res.loc["LOF", W] = roc_auc_score(y_te, -LocalOutlierFactor(n_neighbors=20, novelty=True).fit(X_tr).decision_function(X_te))
        res.loc["PatchCore", W] = roc_auc_score(y_te, pairwise_distances(X_te, k_center(X_tr), metric='euclidean').min(axis=1))
        
        Xt_tr, Xt_te = torch.tensor(X_tr, dtype=torch.float32).to(dev), torch.tensor(X_te, dtype=torch.float32).to(dev)
        ldr = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xt_tr, Xt_tr), batch_size=256, shuffle=True)
        
        # AE
        ae = AE(len(fts)).to(dev); o = torch.optim.Adam(ae.parameters(), lr=1e-3)
        for _ in range(15):
            for b,_ in ldr: o.zero_grad(); nn.MSELoss()(ae(b),b).backward(); o.step()
        with torch.no_grad(): res.loc["Autoencoder",W] = roc_auc_score(y_te, torch.mean((ae(Xt_te)-Xt_te)**2, dim=1).cpu())
        
        # VAE
        vae = VAE(len(fts)).to(dev); o = torch.optim.Adam(vae.parameters(), lr=1e-3)
        for _ in range(15):
            for b,_ in ldr: o.zero_grad(); x,m,lv=vae(b); (nn.MSELoss()(x,b) - 0.0005*torch.sum(1+lv-m.pow(2)-lv.exp())).backward(); o.step()
        with torch.no_grad(): res.loc["VAE",W] = roc_auc_score(y_te, torch.mean((vae(Xt_te)[0]-Xt_te)**2, dim=1).cpu())
        
        # DeepSVDD
        svdd = DeepSVDD(len(fts)).to(dev); o = torch.optim.Adam(svdd.parameters(), lr=1e-3)
        with torch.no_grad(): c = torch.mean(svdd(Xt_tr), dim=0)
        for _ in range(15):
            for b,_ in ldr: o.zero_grad(); torch.mean(torch.sum((svdd(b)-c)**2,dim=1)).backward(); o.step()
        with torch.no_grad(): res.loc["DeepSVDD",W] = roc_auc_score(y_te, torch.sum((svdd(Xt_te)-c)**2, dim=1).cpu())
        
        # TS
        t, s = TSNetwork(len(fts)).to(dev), TSNetwork(len(fts)).to(dev); o = torch.optim.Adam(s.parameters(), lr=1e-3)
        for p in t.parameters(): p.requires_grad=False
        for _ in range(15):
            for b,_ in ldr: o.zero_grad(); nn.MSELoss()(s(b),t(b)).backward(); o.step()
        with torch.no_grad(): res.loc["TeacherStudent",W] = roc_auc_score(y_te, torch.mean((s(Xt_te)-t(Xt_te))**2, dim=1).cpu())
        
        # SSL DAE
        dae = DAE(len(fts)).to(dev); o = torch.optim.Adam(dae.parameters(), lr=1e-3)
        for _ in range(15):
            for b,_ in ldr: o.zero_grad(); nn.MSELoss()(dae(b + 0.5*torch.randn_like(b)),b).backward(); o.step()
        with torch.no_grad(): res.loc["SSL_DAE",W] = roc_auc_score(y_te, torch.mean((dae(Xt_te)-Xt_te)**2, dim=1).cpu())
        
    return res

print("--- Unsupervised AUC (%) [44 Features] ---")
res_44 = run_unsup(feat_cols)
display((res_44 * 100).round(2))

print("\n--- Unsupervised AUC (%) [Top 13 KAN Features] ---")
res_13 = run_unsup(TOP_13_VECTORS)
display((res_13 * 100).round(2))


--- Unsupervised AUC (%) [44 Features] ---


,300,400,500,600,700,800,900,1000
IsolationForest,60.26,73.52,69.15,76.95,82.33,85.65,86.04,86.99
OC-SVM,77.22,82.07,87.91,88.89,94.12,96.55,98.88,98.25
LOF,88.15,91.15,92.96,94.04,95.96,97.74,99.40,99.28
PatchCore,80.45,80.26,84.88,84.83,94.72,89.74,95.83,97.75
Autoencoder,68.69,75.49,79.55,81.41,87.79,85.42,91.74,91.93
VAE,59.57,64.16,65.10,69.64,76.79,79.44,82.94,84.22
DeepSVDD,62.38,74.18,69.00,79.41,75.24,93.22,85.45,95.26
TeacherStudent,72.35,86.07,79.98,84.82,81.03,89.59,92.35,97.64
SSL_DAE,76.61,76.74,83.44,87.12,88.16,91.31,94.20,95.11



--- Unsupervised AUC (%) [Top 13 KAN Features] ---


,300,400,500,600,700,800,900,1000
IsolationForest,78.56,88.19,89.80,93.20,95.36,96.81,98.97,98.11
OC-SVM,90.07,94.16,96.26,97.69,99.41,99.50,99.83,99.70
LOF,94.67,96.36,97.95,98.82,99.14,99.49,99.84,99.75
PatchCore,77.21,91.29,86.33,89.23,96.75,98.65,96.10,98.28
Autoencoder,77.18,88.00,94.40,93.89,96.03,95.23,98.30,98.15
VAE,75.77,81.42,84.26,89.76,92.78,94.87,97.09,97.03
DeepSVDD,88.97,65.40,87.15,88.81,84.92,99.24,88.75,86.29
TeacherStudent,81.43,87.45,84.32,91.74,96.54,98.07,89.53,99.74
SSL_DAE,83.76,90.68,90.93,91.91,98.30,98.65,99.50,98.46


## 6) Supervised Models Sweep (W=300 to 1000) on 13 Features

In [10]:
# 6) Sweep supervised models (including KAN) from W = 300 to 1000 using the 13 feature vectors
results_13F = pd.DataFrame(index=list(get_supervised_models().keys()) + ["KAN"], columns=WINDOWS, dtype=float)

for W in WINDOWS:
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f"features_W{W}.csv"))
    X, y = df[TOP_13_VECTORS].values, df["label"].values
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    fold_accs = {m: [] for m in results_13F.index}
    
    for tr, te in skf.split(X, y):
        sc = MinMaxScaler(); X_tr, X_te = sc.fit_transform(X[tr]), sc.transform(X[te])
        models = get_supervised_models()
        for mname, clf in models.items():
            clf.fit(X_tr, y[tr])
            fold_accs[mname].append(accuracy_score(y[te], clf.predict(X_te)) * 100)
        fold_accs["KAN"].append(train_kan_supervised(X_tr, y[tr], X_te, y[te]))
        
    for m in results_13F.index: results_13F.loc[m, W] = np.mean(fold_accs[m])

print("--- Supervised Models Accuracy (%) [13 Features] ---")
display(results_13F.round(2))


--- Supervised Models Accuracy (%) [13 Features] ---


,300,400,500,600,700,800,900,1000
DT,96.16,97.07,97.12,97.29,97.84,98.49,99.20,99.20
RF,98.19,98.79,99.18,99.29,99.44,99.48,99.78,99.80
SVM,99.00,99.62,99.75,99.82,99.93,99.96,100.00,99.95
NB,89.86,92.27,93.82,96.16,97.15,98.57,99.28,99.60
KNN,98.42,99.44,99.55,99.64,99.90,99.92,99.96,99.95
GBC,98.44,98.99,99.18,99.29,99.37,99.32,99.73,99.65
LR,98.07,98.89,99.28,99.76,99.79,99.88,99.96,99.95
KAN,98.45,98.63,98.34,97.56,97.64,99.09,99.33,99.45
